# HateBERT Fine-tuning for PCL Detection
Fine-tunes HateBERT with weighted loss to handle class imbalance in the Don't Patronize Me! dataset.

## 1. Install Dependencies

In [ ]:
!pip install transformers datasets scikit-learn torch --quiet

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
import ast
import os

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Config

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR       = 'datasets'
TSV_PATH       = os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv')
TRAIN_PATH     = os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv')
DEV_PATH       = os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv')

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = 'GroNLP/hateBERT'
MAX_LEN        = 128     # covers >95% of paragraphs based on EDA (mean=48, median=42)
BATCH_SIZE     = 32
EPOCHS         = 5
LEARNING_RATE  = 2e-5
WARMUP_RATIO   = 0.1     # 10% of steps used for warmup

## 4. Load & Prepare Data

In [ ]:
# Load main tsv
df = pd.read_csv(
    TSV_PATH,
    sep='\t',
    header=None,
    names=['par_id', 'art_id', 'keyword', 'country', 'text', 'label'],
    quoting=3,
    engine='python'
)

# Binary label: {0,1} -> 0 (No PCL), {2,3,4} -> 1 (PCL)
df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0, 1] else 1)

# Load split files — we only need par_id from these
train_ids = pd.read_csv(TRAIN_PATH)['par_id'].tolist()
dev_ids   = pd.read_csv(DEV_PATH)['par_id'].tolist()

# Filter to train and dev splits
train_df = df[df['par_id'].isin(train_ids)].reset_index(drop=True)
dev_df   = df[df['par_id'].isin(dev_ids)].reset_index(drop=True)

print(f'Train size : {len(train_df)} | PCL: {train_df["binary_label"].sum()} ({100*train_df["binary_label"].mean():.1f}%)')
print(f'Dev size   : {len(dev_df)}   | PCL: {dev_df["binary_label"].sum()} ({100*dev_df["binary_label"].mean():.1f}%)')

## 5. Compute Class Weights

In [ ]:
# Weight each class inversely proportional to its frequency
n_samples  = len(train_df)
n_pcl      = train_df['binary_label'].sum()
n_no_pcl   = n_samples - n_pcl

weight_no_pcl = n_samples / (2 * n_no_pcl)
weight_pcl    = n_samples / (2 * n_pcl)

class_weights = torch.tensor([weight_no_pcl, weight_pcl], dtype=torch.float).to(device)
print(f'Class weights -> No PCL: {weight_no_pcl:.4f} | PCL: {weight_pcl:.4f}')

## 6. Dataset & DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = PCLDataset(train_df['text'].tolist(), train_df['binary_label'].tolist(), tokenizer, MAX_LEN)
dev_dataset   = PCLDataset(dev_df['text'].tolist(),   dev_df['binary_label'].tolist(),   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}')

## 7. Model, Optimiser & Scheduler

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

# Weighted cross-entropy loss — penalises misclassifying PCL more heavily
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimiser = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler     = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)

print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

## 8. Training Loop

In [ ]:
def train_epoch(model, loader, optimiser, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimiser.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1, all_preds, all_labels

In [ ]:
best_f1        = 0
best_model_path = 'best_hatebert_pcl.pt'

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model, train_loader, optimiser, scheduler, loss_fn, device)
    dev_loss, dev_f1, _, _ = evaluate(model, dev_loader, loss_fn, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    # Save best model based on dev F1
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.state_dict(), best_model_path)
        print(f'  >> New best model saved (F1: {best_f1:.4f})')

print(f'\nBest dev F1: {best_f1:.4f}')

## 9. Final Evaluation on Dev Set

In [ ]:
# Load best checkpoint and evaluate
model.load_state_dict(torch.load(best_model_path, map_location=device))
_, dev_f1, preds, labels = evaluate(model, dev_loader, loss_fn, device)

print('=== Final Dev Set Results ===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

## 10. Generate Predictions for Test Set

In [ ]:
# If you have a test set file with par_ids, load and predict here
# Adjust the path and column name as needed

TEST_PATH = os.path.join(DATA_DIR, 'test_semeval_parids.csv')  # adjust if needed

if os.path.exists(TEST_PATH):
    test_ids = pd.read_csv(TEST_PATH)['par_id'].tolist()
    test_df  = df[df['par_id'].isin(test_ids)].reset_index(drop=True)

    # Labels are all -1 since test set labels are held out
    test_dataset = PCLDataset(test_df['text'].tolist(), [-1]*len(test_df), tokenizer, MAX_LEN)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            test_preds.extend(preds)

    # Save predictions
    test_df['prediction'] = test_preds
    test_df[['par_id', 'prediction']].to_csv('test_predictions.csv', index=False)
    print(f'Test predictions saved. PCL predicted: {sum(test_preds)} / {len(test_preds)}')
else:
    print('No test file found — skipping test prediction.')